# Сабмишн: компактный набор поведения

Этот ноутбук обучает выбранный CatBoost на всём `train` и один раз строит предсказания для `test`. Метрики здесь не считаются: у тестовой части нет целевой переменной.

Набор признаков и параметры модели должны быть выбраны в предыдущих исследовательских ноутбуках. Идентификаторы используются только для соединения таблиц и не передаются модели.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from artifacts import configure_logging
from config import CATBOOST_FINAL_PARAMS, RANDOM_SEED, ExperimentConfig
from data import load_metadata, load_or_prepare_events
from feature_sets import build_feature_set
from submission import make_submission

configure_logging()
pd.set_option("display.max_columns", 100)

c:\Users\Kolya\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Сборка признаков

При повторном запуске подготовленные события и поведенческие признаки загружаются из `artifacts`.

In [2]:
data = load_metadata()
events = load_or_prepare_events(data.metadata)
features, feature_spec = build_feature_set(
    "compact_behavior",
    data.metadata,
    data.train,
    events,
)

pd.Series({
    "train_rows": len(data.train),
    "test_rows": len(data.test),
    "feature_rows": len(features),
    "behavior_features": features.shape[1] - 1,
    "random_seed": RANDOM_SEED,
}).to_frame("value")

15:39:43 | INFO | data | Метаданные загружены: train=11091, test=4909, ботов=899
15:39:44 | INFO | artifacts | События внутри окон: кеш, строк=288126, колонок=18
15:39:45 | INFO | feature_sets | Поведенческие наборы загружены из кеша


,value
train_rows,11091
test_rows,4909
feature_rows,16000
behavior_features,60
random_seed,42


## Обучение и сохранение

Результат сохраняется в отдельной папке `submissions/compact_behavior/<дата_и_время>/`. Рядом с файлом предсказаний сохраняются модель, параметры, список признаков и правила предобработки.

In [3]:
final_config = ExperimentConfig(
    name="final_compact_behavior",
    feature_set="compact_behavior",
    model_name="catboost",
    model_params=CATBOOST_FINAL_PARAMS,
    use_frequency_features=True,
    frequency_feature_style="suffix",
    expected_feature_count=66,
    feature_version="compact_behavior_v1",
    compute_mode="cpu",
)

run_dir, submission = make_submission(
    final_config,
    features,
    data.train,
    data.test,
    events,
    data.sample_submission,
    submission_group="compact_behavior",
)

pd.Series({
    "submission_path": str(run_dir / "submission.csv"),
    "rows": len(submission),
    "missing_scores": int(submission["score"].isna().sum()),
    "min_score": submission["score"].min(),
    "max_score": submission["score"].max(),
}).to_frame("value")

0:	total: 228ms	remaining: 4m 33s
100:	total: 5.67s	remaining: 1m 1s
200:	total: 10.5s	remaining: 52s
300:	total: 15s	remaining: 44.9s
400:	total: 19.6s	remaining: 39.1s
500:	total: 24.3s	remaining: 33.9s
600:	total: 28.9s	remaining: 28.8s
700:	total: 33.3s	remaining: 23.7s
800:	total: 37.9s	remaining: 18.9s
900:	total: 42.2s	remaining: 14s
1000:	total: 46.6s	remaining: 9.27s
1100:	total: 51s	remaining: 4.58s


15:40:41 | INFO | submission | Submission сохранён в C:\Users\Kolya\Desktop\ОтборНаСтажки\Авито\TestTask-ClassicML-3-avito-ds-bootcamp\submissions\compact_behavior\20260928_154041\submission.csv


1199:	total: 55.4s	remaining: 0us


,value
submission_path,C:\Users\Kolya\Desktop\ОтборНаСтажки\Авито\Tes...
rows,4909
missing_scores,0
min_score,0.000118
max_score,0.999957


## Результат

Модель обучена на всех 11 091 строках train с 66 признаками. Использованы глубина 3, 1 200 деревьев и единое начальное число генератора 42; обучение прошло на процессоре.

Файл содержит 4 909 строк без пропусков и повторов `cookie_id`. Порядок строк совпадает с официальным примером, оценки конечны и лежат в диапазоне от 0.000118 до 0.999957. Все девять проверок формата пройдены. Артефакты запуска сохранены в `submissions/compact_behavior/20260928_154041/`.

Проверяющая система оценила этот сабмишн в **0,81566** по Precision при Recall не ниже 70%. Это первый рабочий результат, который затем использовался как точка отсчёта для исторических признаков.